# AUGMENT DATA — Le Panier-Sûr

Génération de **N observations synthétiques par espèce** à partir du CSV nettoyé, pour alimenter un modèle de classification par nom.

- **Entrée** : `data/transform/champignons_clean.csv`
- **Sortie** : `data/augmented/champignons_augmented.csv`

## ⚠️ Mise en garde data science

Cette augmentation multiplie artificiellement les lignes. Elle fonctionne **uniquement si** :
1. Le split train/test est fait *après* augmentation et *stratifié par espèce* → le modèle voit chaque espèce à l'entraînement ET au test (mémorisation contrôlée, pas généralisation zero-shot).
2. Le bruit introduit est **léger et réaliste** — pas de nouvelles features, uniquement des features existantes qu'on peut "manquer d'observer".
3. Tu n'espères pas que le modèle reconnaisse une espèce jamais vue. Pour ça, il faut de vraies données supplémentaires (multi-sources).

**Ce que l'augmentation apporte** : robustesse du modèle face à une observation partielle (pas toutes les couleurs vues, taille entre min/max, un seul mois d'observation…).

**Ce qu'elle n'apporte PAS** : de la vraie diversité d'espèces. 219 espèces × 100 = 21 900 lignes, mais toujours 219 classes.

## 1. Chargement

In [8]:
import numpy as np
import pandas as pd

CLEAN_CSV = "../../data/transform/champignons_clean.csv"
df = pd.read_csv(CLEAN_CSV)
print(f"{len(df)} espèces, {len(df.columns)} colonnes")
df.head(2)

217 espèces, 174 colonnes


,nom,statut,chapeau,pores,lames,pied,chair,odeur,saveur,habitat,...,habitat_type_jardins,habitat_type_bois,habitat_type_chenes,habitat_type_hetres,habitat_type_pins,habitat_type_bouleaux,habitat_type_chataigniers,habitat_type_charmes,habitat_type_melezes,habitat_type_bois_morts
0,AGARIC AUGUSTE,Champignon à rejeter,"5 à 25 cm, couvert de mèches brun-roux sur fon...",NaN,"Libres, crème puis gris-rose et enfin brunes.","Blanc, élancé (6 à 20 cm), légèrement en massu...","Blanche, jaunissant lentement à la coupe, rosé...","Agréable, d’amande amère",Douce,"Bois clairs de feuillus ou de conifères, lisiè...",...,0,1,0,0,0,0,0,0,0,0
1,AGARIC DES JACHÈRES,Les excellents champignons,"5 à 15 cm, lisse, blanc puis jaunissant en vie...",NaN,"Libres, serrées, gris-rose très pâle puis brun...","5 à 15 cm, élancé, avec un large anneau blanc ...","Blanche, ferme",Anisée,Douce,"Surtout sous feuillus, clairières, lisières, p...",...,0,0,0,0,0,0,0,0,0,0


## 2. Paramètres d'augmentation

| Paramètre | Rôle |
|---|---|
| `N_SAMPLES` | Nombre d'observations synthétiques par espèce |
| `SIZE_SIGMA_RATIO` | Écart-type de la gaussienne sur les tailles, en fraction de l'intervalle [min, max]. `0.25` → ~95% des tirages dans [min, max] |
| `BINARY_DROPOUT` | Probabilité qu'une feature binaire à 1 soit flippée à 0 (simule une observation incomplète) |
| `SEED` | Reproductibilité |

In [9]:
N_SAMPLES        = 1000
SIZE_SIGMA_RATIO = 0.25
BINARY_DROPOUT   = 0.00   # 5% des 1 deviennent 0 (dropout doux)
SEED             = 42

rng = np.random.default_rng(SEED)

## 3. Identification des colonnes par type

Le CSV augmenté ne contiendra que les **features numériques** + `nom` (label) + `statut` (label alternatif). Les colonnes texte brut (`chapeau`, `pores`, `lames`, `pied`, `chair`, `odeur`, `saveur`, `habitat`, `saison`) sont exclues.

Catégorisation :
- **Tailles** (`*_taille_min_cm`, `*_taille_max_cm`) → échantillonnage gaussien dans [min, max]
- **Saison binaire** (`saison_mois_01` … `saison_mois_12`) → tirer un seul mois parmi les actifs
- **Autres binaires** (`*_couleur_*`, `*_texture_*`, `a_un_*`, `habitat_type_*`…) → dropout léger
- **Labels** (`nom`, `statut`) → copie identique

In [10]:
size_cols    = [c for c in df.columns if c.endswith("_taille_min_cm") or c.endswith("_taille_max_cm")]
season_cols  = [c for c in df.columns if c.startswith("saison_mois_")]

# Colonnes binaires "classiques" : toutes les 0/1 sauf saison (traitée à part)
binary_cols = []
for c in df.columns:
    if c in season_cols or c in size_cols:
        continue
    if df[c].dropna().isin([0, 1]).all() and df[c].dtype != object:
        binary_cols.append(c)

# Seules colonnes non-numériques conservées : nom (label principal) et statut (label alternatif)
LABEL_COLS = ["nom", "statut"]
label_cols = [c for c in LABEL_COLS if c in df.columns]

# Toutes les autres colonnes texte (descriptions brutes) sont droppées
dropped_text_cols = [c for c in df.columns
                     if c not in size_cols + season_cols + binary_cols + label_cols]

print(f"Tailles    : {len(size_cols)}")
print(f"Saison     : {len(season_cols)}")
print(f"Binaires   : {len(binary_cols)}")
print(f"Labels     : {label_cols}")
print(f"Droppées   : {dropped_text_cols}")

Tailles    : 4
Saison     : 12
Binaires   : 147
Labels     : ['nom', 'statut']
Droppées   : ['chapeau', 'pores', 'lames', 'pied', 'chair', 'odeur', 'saveur', 'habitat', 'saison']


## 4. Fonctions d'augmentation

### Tailles — gaussienne tronquée

Pour chaque paire `(min, max)` de l'espèce, on tire :
- **μ** = (min + max) / 2 (centre de l'intervalle)
- **σ** = (max - min) × `SIZE_SIGMA_RATIO`
- Clamp dans `[min, max]` pour rester cohérent.

On tire deux valeurs indépendantes et on garantit `min ≤ max`.

### Binaires — dropout

Chaque `1` est conservé avec probabilité `1 - BINARY_DROPOUT`, sinon flippé à `0`. Les `0` restent toujours `0` (on n'invente pas de features absentes de la description source).

### Saison — un mois observé

Au lieu de conserver les 12 flags, on tire un mois uniforme parmi les mois actifs. L'observation est alors datée de ce mois uniquement. On reconstruit ensuite les 12 colonnes avec un seul `1`.

In [11]:
def sample_size(vmin, vmax, n, rng):
    """n tirages gaussiens tronqués dans [vmin, vmax]."""
    if pd.isna(vmin) or pd.isna(vmax) or vmax <= 0:
        return np.zeros(n)
    mu = (vmin + vmax) / 2
    sigma = max((vmax - vmin) * SIZE_SIGMA_RATIO, 1e-6)
    samples = rng.normal(mu, sigma, n)
    return np.clip(samples, vmin, vmax)

def dropout_binary(values, n, rng):
    """Pour chaque valeur binaire, réplique n fois avec dropout sur les 1."""
    out = np.tile(values, (n, 1))            # shape (n, n_cols)
    mask_ones = out == 1
    flip = rng.random(out.shape) < BINARY_DROPOUT
    out[mask_ones & flip] = 0
    return out

def sample_season_month(active_months, n, rng):
    """Tire n mois parmi les actifs. Renvoie un array (n, 12) avec un seul 1 par ligne."""
    result = np.zeros((n, 12), dtype=int)
    if len(active_months) == 0:
        return result
    picks = rng.choice(active_months, size=n)  # valeurs dans 1..12
    result[np.arange(n), picks - 1] = 1
    return result

## 5. Boucle d'augmentation

Pour chaque espèce du dataset, on génère `N_SAMPLES` lignes en appliquant les trois transformations.

In [12]:
augmented_rows = []

for _, row in df.iterrows():
    # 1. Labels : recopie à l'identique (nom, statut)
    base = {col: [row[col]] * N_SAMPLES for col in label_cols}

    # 2. Tailles : gaussienne
    sizes = {}
    parts_with_size = {c.rsplit("_taille_", 1)[0] for c in size_cols}
    for part in parts_with_size:
        vmin = row.get(f"{part}_taille_min_cm", 0)
        vmax = row.get(f"{part}_taille_max_cm", 0)
        s1 = sample_size(vmin, vmax, N_SAMPLES, rng)
        s2 = sample_size(vmin, vmax, N_SAMPLES, rng)
        lo = np.minimum(s1, s2)
        hi = np.maximum(s1, s2)
        sizes[f"{part}_taille_min_cm"] = lo
        sizes[f"{part}_taille_max_cm"] = hi

    # 3. Binaires : dropout
    bin_values = row[binary_cols].to_numpy().astype(int)
    bin_aug = dropout_binary(bin_values, N_SAMPLES, rng)
    bin_dict = {col: bin_aug[:, i] for i, col in enumerate(binary_cols)}

    # 4. Saison : un seul mois tiré parmi les actifs
    active = np.array([i + 1 for i, c in enumerate(season_cols) if row[c] == 1])
    season_aug = sample_season_month(active, N_SAMPLES, rng)
    season_dict = {col: season_aug[:, i] for i, col in enumerate(season_cols)}

    # Assemblage
    chunk = pd.DataFrame({**base, **sizes, **bin_dict, **season_dict})
    augmented_rows.append(chunk)

# Ordre des colonnes : labels d'abord, puis tailles, puis binaires, puis saison
ordered_cols = label_cols + size_cols + binary_cols + season_cols
df_aug = pd.concat(augmented_rows, ignore_index=True)[ordered_cols]

print(f"Avant : {len(df)} espèces × {len(df.columns)} colonnes")
print(f"Après : {len(df_aug)} lignes × {len(df_aug.columns)} colonnes ({N_SAMPLES}× par espèce)")
df_aug.head(3)

Avant : 217 espèces × 174 colonnes
Après : 217000 lignes × 165 colonnes (1000× par espèce)


,nom,statut,chapeau_taille_min_cm,chapeau_taille_max_cm,pied_taille_min_cm,pied_taille_max_cm,a_un_chapeau,a_des_pores,a_des_lames,a_un_pied,...,saison_mois_03,saison_mois_04,saison_mois_05,saison_mois_06,saison_mois_07,saison_mois_08,saison_mois_09,saison_mois_10,saison_mois_11,saison_mois_12
0,AGARIC AUGUSTE,Champignon à rejeter,12.740245,21.245113,12.792511,14.066510,1,0,1,1,...,0,0,0,0,0,0,0,1,0,0
1,AGARIC AUGUSTE,Champignon à rejeter,11.670611,18.438461,9.360056,10.447496,1,0,1,1,...,0,0,0,0,0,0,0,1,0,0
2,AGARIC AUGUSTE,Champignon à rejeter,17.170049,24.830637,11.549344,15.626579,1,0,1,1,...,0,0,0,0,0,1,0,0,0,0


## 6. Sanity checks

Validation rapide que l'augmentation a fait ce qu'on attend.

In [13]:
# (a) Chaque espèce a bien N_SAMPLES lignes
counts = df_aug["nom"].value_counts()
print(f"Lignes par espèce : min={counts.min()}, max={counts.max()} (attendu : {N_SAMPLES})")

# (b) Les tailles varient au sein d'une espèce
size_check_col = "chapeau_taille_min_cm" if "chapeau_taille_min_cm" in df_aug.columns else size_cols[0]
sample_species = df_aug["nom"].iloc[0]
sizes_species = df_aug.loc[df_aug["nom"] == sample_species, size_check_col]
print(f"\n{sample_species} — {size_check_col} :")
print(f"  min={sizes_species.min():.2f}  max={sizes_species.max():.2f}  std={sizes_species.std():.2f}")

# (c) La saison ne contient qu'un seul 1 par ligne
if season_cols:
    season_sum = df_aug[season_cols].sum(axis=1)
    print(f"\nSaison — nb de mois actifs par ligne : unique={sorted(season_sum.unique())}")

# (d) Taux moyen de 1 avant/après sur les binaires (doit avoir légèrement baissé)
if binary_cols:
    rate_before = df[binary_cols].mean().mean()
    rate_after = df_aug[binary_cols].mean().mean()
    print(f"\nBinaires — taux moyen de 1 : {rate_before:.3f} → {rate_after:.3f} (dropout {BINARY_DROPOUT})")

Lignes par espèce : min=1000, max=1000 (attendu : 1000)

AGARIC AUGUSTE — chapeau_taille_min_cm :
  min=5.00  max=24.44  std=3.92

Saison — nb de mois actifs par ligne : unique=[np.int64(0), np.int64(1)]

Binaires — taux moyen de 1 : 0.113 → 0.113 (dropout 0.0)


## 7. Export

Les observations sont **conservées groupées par espèce** (pas de shuffle) — chaque espèce occupe `N_SAMPLES` lignes consécutives. Le shuffle se fera au moment du `train_test_split` côté modélisation.

In [14]:
import os

OUT_DIR = "../../data/augmented"
os.makedirs(OUT_DIR, exist_ok=True)
OUT_CSV = f"{OUT_DIR}/champignons_augmented.csv"

df_aug.to_csv(OUT_CSV, index=False)
print(f"{len(df_aug)} lignes, {len(df_aug.columns)} colonnes → {OUT_CSV}")
print(f"Ordre conservé : {N_SAMPLES} lignes consécutives par espèce.")

217000 lignes, 165 colonnes → ../../data/augmented/champignons_augmented.csv
Ordre conservé : 1000 lignes consécutives par espèce.
